# Git Basics for Machine Learning

**Soft prerequisite** -- you do not need to be an expert before the course starts. We will reinforce Git skills from Session 1 onward. This notebook gives you a head start.

**What this covers:** version control fundamentals, the core Git workflow (init, add, commit, push, pull), branching, and `.gitignore` best practices for ML projects.

**Who it is for:** students who have never used Git, or who have used it casually and want a structured refresher.

**Estimated time:** 30--45 minutes.

---
## 1. What Is Version Control and Why It Matters for ML

Version control tracks changes to files over time so you can recall any previous state. Think of it as an unlimited undo history that also lets multiple people work on the same project simultaneously.

**Why ML practitioners care:**

- **Reproducibility.** You need to know which code produced which results. A commit hash pins your code to a precise snapshot.
- **Experimentation.** Branches let you try different model architectures or hyperparameters without breaking what already works.
- **Collaboration.** Research teams share notebooks, training scripts, and config files. Git keeps everyone in sync.
- **Audit trail.** When a model behaves unexpectedly, `git log` and `git diff` help you find what changed and when.

Git is the industry standard. GitHub, GitLab, and Bitbucket are hosting platforms built around it.

---
## 2. Core Git Concepts

| Concept | Definition |
|---------|------------|
| **Repository (repo)** | A directory tracked by Git. Contains your files plus a hidden `.git/` folder with the full history. |
| **Commit** | A snapshot of the repo at a point in time. Each commit has a unique SHA hash, a message, an author, and a timestamp. |
| **Branch** | A movable pointer to a commit. The default branch is usually called `main`. Branches let you develop features in isolation. |
| **Remote** | A copy of the repo hosted elsewhere (e.g., on GitHub). The default remote is called `origin`. |
| **Staging area (index)** | A buffer between your working directory and the next commit. You explicitly choose which changes to include. |

The basic flow:

```
Working Directory  --git add-->  Staging Area  --git commit-->  Local Repo  --git push-->  Remote Repo
```

---
## 3. Basic Workflow: init, add, commit, status, log

Let's create a practice repository from scratch.

In [ ]:
%%bash
# Clean up any previous practice directory, then create a fresh one
rm -rf /tmp/git-practice
mkdir /tmp/git-practice
cd /tmp/git-practice

# Initialize a new Git repository
git init

# Configure user info (local to this repo only)
git config user.name "ML Student"
git config user.email "student@example.com"

In [ ]:
%%bash
cd /tmp/git-practice

# Create a file
echo "# My ML Project" > README.md
echo "print('hello world')" > train.py

# Check the status -- both files are untracked
git status

In [ ]:
%%bash
cd /tmp/git-practice

# Stage files (add them to the index)
git add README.md train.py

# Commit with a descriptive message
git commit -m "Initial commit: add README and training script"

# View the commit log
git log --oneline

In [ ]:
%%bash
cd /tmp/git-practice

# Modify an existing file
echo "import torch" >> train.py
echo "model = torch.nn.Linear(10, 1)" >> train.py

# See what changed (unstaged diff)
git diff train.py

In [ ]:
%%bash
cd /tmp/git-practice

# Stage and commit the update
git add train.py
git commit -m "Add PyTorch model definition"

# Log now shows two commits
git log --oneline

### Exercise 3.1

In the `/tmp/git-practice` repo:

1. Create a file called `evaluate.py` with the content `print('evaluating model')`.
2. Stage it, commit it with the message `"Add evaluation script"`.
3. Run `git log --oneline` to verify you now have three commits.

In [ ]:
%%bash
cd /tmp/git-practice

# Your solution here


---
## 4. Working with Remotes: clone, push, pull

A **remote** is a version of your repo hosted on a server (GitHub, GitLab, etc.).

| Command | What it does |
|---------|--------------|
| `git clone <url>` | Download a remote repo and set up tracking automatically. |
| `git remote -v` | List configured remotes. |
| `git push origin main` | Upload local commits on `main` to the remote called `origin`. |
| `git pull origin main` | Download and merge remote changes into your local `main`. |

### Typical workflow

```bash
# 1. Clone a repo (do this once)
git clone https://github.com/user/repo.git
cd repo

# 2. Make changes, commit locally
git add .
git commit -m "Improve data loader"

# 3. Push to remote
git push origin main

# 4. Later, pull teammates' changes
git pull origin main
```

In [ ]:
%%bash
# Simulate a remote by creating a bare repo, then clone it
rm -rf /tmp/git-remote-demo /tmp/git-clone-demo

# Create a "remote" (bare repository)
git init --bare /tmp/git-remote-demo

# Clone it into a working directory
git clone /tmp/git-remote-demo /tmp/git-clone-demo

cd /tmp/git-clone-demo
git config user.name "ML Student"
git config user.email "student@example.com"

# Show the remote
git remote -v

In [ ]:
%%bash
cd /tmp/git-clone-demo

# Create a file, commit, and push
echo "learning_rate = 0.001" > config.py
git add config.py
git commit -m "Add training config"
git push origin main

echo "--- Push succeeded ---"

### Exercise 4.1

1. In `/tmp/git-clone-demo`, add a file `data.py` with the line `DATASET = 'cifar10'`.
2. Commit and push it to origin.
3. Verify with `git log --oneline` that you have two commits.

In [ ]:
%%bash
cd /tmp/git-clone-demo

# Your solution here


---
## 5. Branches: create, switch, merge

Branches let you work on a feature or experiment without affecting the main line of development.

| Command | What it does |
|---------|--------------|
| `git branch` | List branches. The current one has a `*`. |
| `git branch <name>` | Create a new branch (does not switch to it). |
| `git switch <name>` | Switch to an existing branch. (Older: `git checkout <name>`.) |
| `git switch -c <name>` | Create and switch in one step. |
| `git merge <name>` | Merge `<name>` into the current branch. |

### Typical branch workflow

```
main:           A---B---C-------F  (merge commit)
                         \     /
feature/aug:              D---E
```

1. Create a branch for your experiment.
2. Make commits on that branch.
3. When the experiment works, merge it back into `main`.

In [ ]:
%%bash
cd /tmp/git-practice

# Create and switch to a new branch
git switch -c feature/augmentation

# Make a change on the feature branch
echo "from torchvision import transforms" > augment.py
git add augment.py
git commit -m "Add data augmentation module"

# Show branches (* marks current)
git branch

In [ ]:
%%bash
cd /tmp/git-practice

# Switch back to main and merge the feature branch
git switch main
git merge feature/augmentation -m "Merge feature/augmentation into main"

# Verify augment.py is now on main
git log --oneline --graph

### Exercise 5.1

1. In `/tmp/git-practice`, create a branch called `experiment/dropout`.
2. On that branch, create a file `dropout.py` with the content `DROPOUT_RATE = 0.3`.
3. Commit it, switch back to `main`, and merge.
4. Run `git log --oneline --graph` to see the history.

In [ ]:
%%bash
cd /tmp/git-practice

# Your solution here


---
## 6. .gitignore -- Keeping Your Repo Clean

ML projects generate files that should **not** be committed: large datasets, model checkpoints, cached bytecode, environment-specific configs. A `.gitignore` file tells Git to ignore matching paths.

### Recommended `.gitignore` for ML projects

```gitignore
# Data files
data/
*.csv
*.parquet
*.h5
*.hdf5

# Model checkpoints
*.pt
*.pth
*.ckpt
*.pkl
checkpoints/

# Python cache
__pycache__/
*.pyc

# Jupyter checkpoints
.ipynb_checkpoints/

# Environment
.env
venv/
.venv/

# IDE
.vscode/
.idea/

# OS
.DS_Store
Thumbs.db

# Weights & Biases
wandb/
```

**Rule of thumb:** if a file is generated or can be downloaded, it does not belong in version control.

In [ ]:
%%bash
cd /tmp/git-practice

# Create a .gitignore
cat > .gitignore << 'EOF'
__pycache__/
*.pyc
*.pt
*.pth
data/
.ipynb_checkpoints/
wandb/
EOF

# Create some files that should be ignored
mkdir -p __pycache__
echo "cache" > __pycache__/train.cpython-310.pyc
echo "weights" > model.pt

# Status: .gitignore is new, but cached/model files are NOT listed
git status

In [ ]:
%%bash
cd /tmp/git-practice

git add .gitignore
git commit -m "Add .gitignore for ML project"
git log --oneline

### Exercise 6.1

1. Add `*.csv` and `checkpoints/` to the `.gitignore` in `/tmp/git-practice`.
2. Create a file `results.csv` and a directory `checkpoints/` with a file inside.
3. Run `git status` and confirm that neither appears as untracked.

In [ ]:
%%bash
cd /tmp/git-practice

# Your solution here


---
## 7. GitHub Desktop -- A GUI Alternative

If you prefer a graphical interface, [GitHub Desktop](https://desktop.github.com/) lets you perform all the operations above (clone, commit, branch, push, pull, merge) through a point-and-click UI.

**When it helps:**
- Visualizing diffs and history.
- Resolving merge conflicts with a side-by-side view.
- Getting started quickly without memorizing commands.

**Our recommendation:** learn the CLI commands first so you understand what is happening, then use a GUI if it speeds up your workflow. During the course we will use the command line.

---
## 8. Working with Two Repos (Course Workflow)

In this course you will work with **two separate repositories**:

| Repo | URL pattern | Your access | Purpose |
|------|------------|-------------|---------|
| **Course repo** | `ai-2026-course` | **Read-only** | Lesson notebooks, homework templates, datasets. Pull updates before each session. |
| **Your student repo** | `ai-2026-<your-name>` | **Read-write** | All your work: homework, projects, notes. You push here. |

### One-time setup (already done for you)

```bash
git clone https://github.com/unrn-ai-2026/ai-2026-course.git
git clone https://github.com/unrn-ai-2026/ai-2026-<your-name>.git
```

### Before each session: pull latest materials

```bash
cd ai-2026-course
git pull origin main
```

This fetches any new notebooks, homework templates, or datasets the instructor has published since the last session.

### Your daily work: in your student repo

```bash
cd ai-2026-<your-name>

# ... edit files ...

git add homework/session-01/homework.ipynb
git commit -m "Complete session 1 homework"
git push origin main
```

**Key rule:** never modify files inside `ai-2026-course` — that repo is read-only for you. Copy templates into your student repo if you need to edit them.

---
## 9. GitHub Issues — Receiving Feedback and Communicating

**Issues** are GitHub's built-in communication tool — think of them as task tickets attached to a repository. Each Issue has a title, a description, a comment thread, and a status (open or closed).

### How we use Issues in this course

- **AI-graded homework feedback** arrives as an Issue on your student repo. After you push a homework notebook, an automated grader reviews it and opens an Issue with detailed feedback — what you got right, what needs improvement, and your score.
- **You can comment on an Issue** to ask clarifying questions or appeal a feedback point. The instructor monitors these comments.
- **You create Issues** when you need something — for example, using a template to request attendance recovery or to submit a feedback appeal.

### Navigating Issues on your repo

1. **View Issues.** Go to your student repo on GitHub and click the **Issues** tab at the top. You will see a list of all open Issues.
2. **Read an Issue.** Click on any Issue title to see its full description and comment thread.
3. **Comment on an Issue.** Scroll to the bottom of an Issue page and type your response in the comment box. Click **Comment** to post.
4. **Create a new Issue.** Click the green **New issue** button. If templates are available, pick the one that matches your need (e.g., "Attendance Recovery" or "Feedback Appeal") and fill in the required fields.
5. **Close an Issue.** Once the matter is resolved, click **Close issue** at the bottom. Closed Issues remain accessible in the "Closed" filter.

> **Notifications.** You will receive an email notification when a new Issue is created on your repo. Make sure your GitHub notification settings are configured (see Section 11 below).

---
## 10. Pull Requests — Final Project Submission

A **Pull Request** (PR) is a request to merge one branch into another. It is the standard mechanism for code review in professional software development — and it is how you will submit your final project in this course.

### Why Pull Requests?

- They give the instructor a clear view of everything you changed.
- The instructor can leave line-by-line comments on your code.
- You can push additional commits to address feedback without starting over.
- Once approved, you merge the PR and the work becomes part of `main`.

### Final project workflow

```bash
# Create and switch to the final-project branch
git switch -c final-project

# Work on your project...
git add .
git commit -m "Add model training script"
git push origin final-project

# Then on GitHub: create a Pull Request from final-project -> main
```

### What happens in the PR

1. **You create the PR** on GitHub: go to your repo, click **Pull requests** > **New pull request**, select `final-project` as the source branch and `main` as the target, then click **Create pull request**.
2. **The instructor reviews your code.** You will see comments on specific lines or general comments on the PR.
3. **You address feedback** by making changes locally, committing, and pushing to the same `final-project` branch — the PR updates automatically.
4. **Once approved, you merge the PR** by clicking the **Merge pull request** button. Your final project code is now on `main`.

---
## 11. GitHub Notifications — Staying on Top of Things

You will receive automated feedback, instructor comments, and PR reviews through GitHub. Missing a notification means missing important information. Take two minutes to configure this now.

### Recommended settings

1. Go to [github.com/settings/notifications](https://github.com/settings/notifications).
2. Under **"Participating, @mentions and custom"**, make sure **Email** is checked. This ensures you get an email whenever someone comments on your Issue or PR, or mentions you.
3. Optionally enable **Web and mobile** notifications as well.

### Checking notifications

- Visit [github.com/notifications](https://github.com/notifications) to see all your unread notifications in one place.
- You can also install the **GitHub mobile app** (iOS / Android) to get push notifications on your phone.

**Tip:** if your inbox gets noisy later in the semester, you can fine-tune notifications per repository — but for now, keep everything on so you do not miss feedback.

---
## Self-Assessment

Answer these without looking back. If you are unsure about any of them, revisit the relevant section.

1. **What is the difference between `git add` and `git commit`?** Explain the role of the staging area.

2. **You made changes to three files but only want to commit two of them.** What commands would you use?

3. **A teammate pushed changes to `main` on GitHub while you were working locally.** What command do you run before pushing your own commits, and why?

4. **Your ML project has a `data/` folder with 2 GB of training images.** Should you commit it? If not, what do you do instead?

5. **You received AI feedback on your homework as a GitHub Issue. You disagree with one point.** What do you do?

6. **Your final project is ready to submit.** Describe the branch + PR workflow step by step.

---
## References

- [Pro Git Book (free)](https://git-scm.com/book/en/v2) -- The definitive reference. Chapters 1-3 cover everything in this notebook and more.
- [GitHub Git Cheat Sheet (PDF)](https://education.github.com/git-cheat-sheet-education.pdf) -- One-page reference to pin next to your monitor.
- [Oh Shit, Git!?!](https://ohshitgit.com/) -- Practical recipes for fixing common Git mistakes.
- [GitHub Issues docs](https://docs.github.com/en/issues) -- Official guide to Issues.
- [GitHub Pull Requests docs](https://docs.github.com/en/pull-requests) -- Official guide to PRs.